## Fetching Articles From Wikipedia on topics(e.g, Machine Learning)

In [1]:
from langchain_core.documents import Document
import wikipedia

Topics = [
    'Machine Learning',
    'Deep Learning',
    'Natural Language Processing',
    'Artificial Intelligence',
]

documents = []

for topic in Topics:
    page = wikipedia.page(topic, auto_suggest=False)

    doc = Document(
    page_content=page.content[:5000],
    metadata={"source": page.url, "title": page.title}
    )

    documents.append(doc)

    print(f"Document Loaded: {topic} ({len(page.content[:5000])} chars)")

print(f"Total Documents Loaded: {len(documents)}")


Document Loaded: Machine Learning (5000 chars)
Document Loaded: Deep Learning (5000 chars)
Document Loaded: Natural Language Processing (5000 chars)
Document Loaded: Artificial Intelligence (5000 chars)
Total Documents Loaded: 4


## Spitting Doucments into Chunks

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = splitter.split_documents(documents)

print(f"Original documents: {len(documents)}")
print(f"After Splitting: {len(chunks)} chunks")
print(f"\nExample (first one):")
print("-" * 50)
print(chunks[0].page_content)
print("-" * 50)
print(f"Source: {chunks[0].metadata['source']}")



Original documents: 4
After Splitting: 61 chunks

Example (first one):
--------------------------------------------------
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions. Within a subdiscipline of machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
--------------------------------------------------
Source: https://en.wikipedia.org/wiki/Machine_learning


## Create Embeddings and Build the Vector Store

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {"device": "cpu" }
)

vector_store = FAISS.from_documents(chunks, embeddings)

vector_store.save_local("faiss_index")

print(f"Vector Store built and saved!")
print(f"Indexed from {len(chunks)} to {len(documents)} documents")
print(f"Quick Test Search: 'Define Deep Learning.'")
result = vector_store.similarity_search("Define Deep Learning.", k=2)

for i, r in enumerate(result):
    print(f"Result {i+1}: {r.page_content[:200]}...")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector Store built and saved!
Indexed from 61 to 4 documents
Quick Test Search: 'Define Deep Learning.'
Result 1: Fundamentally, deep learning refers to a class of machine learning algorithms in which a hierarchy of layers is used to transform input data into a progressively more abstract and composite representa...
Result 2: In machine learning, deep learning (DL) focuses on utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning. The field takes inspiration ...


## Set Up the LLM

In [5]:

from langchain_groq import ChatGroq
from getpass import getpass
import os

api_key = getpass("Enter GROQ API Key: ")
os.environ["api_key"] = api_key


llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0.3,                   
    max_tokens=1024,                   
    groq_api_key=api_key           
)

print("Testing LLM Connection...")
response = llm.invoke("What is Machine Learning in one sentence?")
print(response)

Testing LLM Connection...
content='Machine Learning is a subset of artificial intelligence that involves training algorithms to learn from data and make predictions, decisions, or take actions without being explicitly programmed.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 43, 'total_tokens': 74, 'completion_time': 0.083891271, 'completion_tokens_details': None, 'prompt_time': 0.003980468, 'prompt_tokens_details': None, 'queue_time': 0.022950077, 'total_time': 0.087871739}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019dc55e-fee5-7440-97ab-e0e7421c1bc1-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 43, 'output_tokens': 31, 'total_tokens': 74}


## Build the RAG Chain with Memory

In [12]:
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate

memory = ConversationBufferWindowMemory(
    k=5,
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False 
)

print("Testing the full RAG pipeline...")
result = qa_chain.invoke({"question": "What are AI Agents?"})
print(f"What do you know about Autonomous systems?")
print(f"\nAnswer: {result['answer']}")
print("\nsources used: ")
for doc in result['source_documents']:
    print(f" - {doc.metadata.get('title', 'Unknown')}")


Testing the full RAG pipeline...
What do you know about Autonomous systems?

Answer: According to the context, an "agent" is any entity (artificial or not) that perceives and takes actions in the world. In the context of AI, an AI agent is likely a computational system that perceives its environment and takes actions to achieve defined goals, using learning and intelligence.

sources used: 
 - Artificial intelligence
 - Artificial intelligence
 - Artificial intelligence


## Testing with Memory

In [7]:
q1 = qa_chain.invoke({"question": "What is the difference between supervised and unsupervised learnig?"})
print(f"Q1: What is the difference between supervised and unsupervised learning?")
print(f"A1: {q1['answer']}")

print("-" * 50)
print("\n")

q2 = qa_chain.invoke({"question": "Also mention some applications of both in real world."})
print("Q2: Also mention some applications of both in real world.")
print(f"A2: {q2['answer']}")

Q1: What is the difference between supervised and unsupervised learning?
A1: The main difference between supervised and unsupervised learning is the objective and the type of data used.

Supervised learning involves training algorithms on labeled data, where the correct output is already known. The objective of supervised learning is to learn a mapping between input data and the corresponding output labels, and it is commonly used for tasks such as classification and regression.

Unsupervised learning, on the other hand, involves training algorithms on unlabeled data, where the correct output is not known. The objective of unsupervised learning is to discover patterns, relationships, or groupings in the data, and it is commonly used for tasks such as clustering, dimensionality reduction, and association rule learning.

In other words, supervised learning is used when you have labeled data and want to make predictions, while unsupervised learning is used when you have unlabeled data and